In [ ]:
# Cluster task. Connects to Lakebase with a Databricks-minted OAuth token (no stored password).
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)
from state_store import StateStore


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


db_instance = widget("db_instance", "lakefed-ingest-mt")
db_name = widget("db_name", "lakefed_ingest_mt")
store = StateStore.connect_lakebase(db_instance, db_name)

In [ ]:
# Insert example cohort rows into the Lakebase control table (one row per tenant database x table).
# Prereqs per table: CT enabled, PK present, ALLOW_SNAPSHOT_ISOLATION ON,
# CHANGE_RETENTION > expected seed time. Source reads use Spark JDBC with the per-database
# credential secret named in `secret_key` (JSON {"user", "password"}) - no UC connection.
# select_list must list explicit columns (`*` is not supported with Change Tracking).
cols = ["job_name", "task_collection", "src_type", "src_host", "src_port", "secret_key",
        "src_database", "src_schema", "src_table", "sink_catalog", "sink_schema", "sink_table",
        "primary_key", "load_type", "select_list", "use_remote_query", "ct_enabled",
        "tier", "priority", "task_enabled",
        # Partitioned seed (large tables): size-bounded range partitions on a numeric/date/datetime
        # column, loaded `partition_concurrency` at a time. Leave load_partitioned False for small tables.
        "load_partitioned", "partition_col", "partition_size_mb"]
rows = [
    ("lakefed_ingest_mt", "example_cohort", "sqlserver", "sqlserver-01.example.com", 1433,
     "tenant-db-001", "tenant_db_001", "dbo", "orders", "main", "bronze_tenant_db_001", "orders",
     "order_id", "ct", "order_id, customer_id, status, updated_at", False, True, "gold", 1, True,
     True, "order_id", 512),
    ("lakefed_ingest_mt", "example_cohort", "sqlserver", "sqlserver-02.example.com", 1433,
     "tenant-db-002", "tenant_db_002", "dbo", "orders", "main", "bronze_tenant_db_002", "orders",
     "order_id", "ct", "order_id, customer_id, status, updated_at", False, True, "gold", 1, True,
     False, None, None),
]
placeholders = ", ".join(["%s"] * len(cols))
with store.connection.cursor() as cur:
    cur.executemany(
        f"insert into lakefed_ingest_mt.control ({', '.join(cols)}) values ({placeholders})", rows)
print(f"inserted {len(rows)} control rows")